## 1. Importação das bibliotecas

In [ ]:
# Instala o pacote watermark
!pip install -q -U watermark

In [ ]:
# Instala o Numpy
!pip install -q numpy==2.3.2

In [ ]:
#importa a biblioteca Numpy
import numpy as np

In [ ]:
# Importa bibliotecas complementares
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Definir uma semente para reprodutibilidade dos resultados
np.random.seed(42)

In [ ]:
%reload_ext watermark
%watermark -a "Valter de Pauli Netto"

In [ ]:
%watermark --iversions

## 2. Função Para a Geração dos Dados Fictícios

Criando um conjunto de dados com 500 usuários. Cada usuário será associado a 4 métricas.
- visitas: Número de vezes que o usuário visitou o site no mês.
- tempo_no_site: Tempo total em minutos que o usuário passou no site.
- itens_no_carrinho: Número de itens que o usuário adicionou no carrinho.
- valor_compra: O valor total em R$ da compra realizada pelo usuário no mês.

In [ ]:
# Definir o número de usuários
num_usuarios = 500

In [ ]:
# Gerar o número de visitas (entre 1 e 50)
visitas = np.random.randint(1, 51, size = num_usuarios)

In [ ]:
# Gerar o tempo no site (distribuição normal, correlacionado com as visitas)
# Média de 20 min, desvio padrão de 5, com um bônus por visita
tempo_no_site = np.random.normal(loc = 20, scale = 5, size = num_usuarios) + (visitas * 0.5)
tempo_no_site = np.round(tempo_no_site, 2) # Arredonda para duas casas decimais

In [ ]:
# Gerar o número de itens no carrinho (depende das visitas e do tempo)
# Usuários que visitam mais e passam mais tempo, tendem a adicionar mais itens
itens_no_carrinho = np.random.randint(0, 8, size = num_usuarios) + (visitas // 10)

In [ ]:
# Garante que o tempo no site também influencie positivamente
itens_no_carrinho = (itens_no_carrinho + (tempo_no_site // 15)).astype(int)

In [ ]:
# Gerar o valor da compra (correlacionado com os itens do carrinho)
# Preço médio por item de R$35,00 com alguma variação aleatória
valor_compra = (itens_no_carrinho * 35) + np.random.normal(loc = 0, scale = 10, size = num_usuarios)

In [ ]:
type(valor_compra)

In [ ]:
type(itens_no_carrinho)


In [ ]:
# Caso não houver itens no carrinho, o valor da compra deve ser 0
valor_compra[itens_no_carrinho == 0] = 0
valor_compra[valor_compra < 0] = 0 # Corrige valores negativos
valor_compra = np.round(valor_compra, 2)

In [ ]:
# Unindo tudo em uma unica matriz (ndarray)
# Cada linha representa um usuário, cada coluna uma métrica
dados_ecommerce = np.column_stack((visitas, tempo_no_site, itens_no_carrinho, valor_compra))

In [ ]:
print("\nShape da massa de dados:", dados_ecommerce.shape)
print("\nExemplo dos 5 primeiros usuários (linhas):")
print("\nColunas: [Visitas, Tempo no Site (min), Itens no Carrinho, Valor da Compra (R$)]\n")
print(dados_ecommerce [:5])

## 3. Análise Estatística Descritiva


In [ ]:
# Separando as colunas para facilitar a leitura do código
visitas_col = dados_ecommerce[:, 0]
tempo_col = dados_ecommerce[:, 1]
itens_col = dados_ecommerce[:, 2]
valor_col = dados_ecommerce[:, 3]

print(" --- ANÁLISE ESTATÍSTICA GERAL --- ")

# Média
media_visitas = np.mean(visitas_col)
media_tempo = np.mean(tempo_col)
media_itens = np.mean(itens_col)
media_valor = np.mean(valor_col)

print(f"\nMédia de Visitas: {media_visitas:.2f}")
print(f"Média de Tempo no Site: {media_tempo:.2f}")
print(f"Média de Itens no Carrinho: {media_itens:.2f}")
print(f"Média de Valor de Compra (Ticket Médio): R$ {media_valor:.2f}")

# Mediana (Valor central, menos sensível a outliers)
mediana_valor = np.median(valor_col)
print(f"\nMediana do Valor de Compra: R$ {mediana_valor:.2f}")

# Desvio Padrão (Dispersão dos Dados)
std_valor = np.std(valor_col)
print(f"\nDesvio Padrão do Valor da Compra: R$ {std_valor:.2f}")


# Valores Máximos e Mínimos
max_valor = np.max(valor_col)
min_valor = np.min(valor_col[valor_col > 0]) # Mínimos apenas entre quem comprou
print(f"\nMaior Valor de Compra: R$ {max_valor:.2f}")
print(f"Menos Valor de Compra: R$ {min_valor:.2f}")

## 3.1 Visualização gráfica

In [ ]:
# Separando colunos
visitas_col = dados_ecommerce[:, 0]
tempo_col = dados_ecommerce[:, 1]
itens_col = dados_ecommerce[:, 2]
valor_col = dados_ecommerce[:, 3]

# Calculando as estatísticas
media_valor = np.mean(valor_col)
mediana_valor = np.median(valor_col)
std_valor = np.std(valor_col)

# --- GRÁFICO ---
plt.figure(figsize = (12, 5))
plt.hist(valor_col, bins = 30, color = 'skyblue', edgecolor = 'black', alpha = 0.7)
plt.axvline(media_valor, color = 'red', linestyle = '--', linewidth = 2, label = f'Média = R$ {media_valor:.2f}' )
plt.axvline(mediana_valor, color = 'yellow', linestyle = '--', linewidth = 2, label = f'Mediana = R$ {mediana_valor:.2f}' )
plt.axvline(media_valor + std_valor, color = 'orange', linestyle = ':', linewidth = 2, label = f'+1 DP = R$ {media_valor + std_valor:.2f}')
plt.axvline(media_valor - std_valor, color = 'orange', linestyle = ':', linewidth = 2, label = f'-1 DP = R$ {media_valor - std_valor:.2f}')
plt.title('Distribuição dos Valores de Compra')
plt.xlabel('Valor de Compra (R$)')
plt.ylabel('Frequência')
plt.legend()
plt.grid(alpha = 0.3)
plt.show()

## 4. Segmentação e Análise de Clientes

In [ ]:
# Filtro booleano para clientes com compra maior que R$ 250,00
clientes_alto_valor = dados_ecommerce[dados_ecommerce[:, 3] > 250]

print("\n--- ANÁLISE: CLIENTE DE ALTO VALOR (compras maiores que R$ 250,00) ---\n")
print(f"Número de clientes de alto valor: {cliente_alto_valor.shape[0]}")

# Estatística deste segmento
media_visitas_alto_valor = np.mean(clientes_alto_valor[:, 0])
media_tempo_alto_valor = np.mean(clientes_alto_valor[:, 1])

print(f"Média de visitas desses clientes: {media_visitas_alto_valor:.2f}")
print(f"Média de tempo no site desses clientes: {media_tempo_alto_valor:.2f}")

In [ ]:
# Filtro para visitantes que não compram
visitantes_sem_compra = dados_ecommerce[dados_ecommerce[:, 3] == 0]

print("\n--- ANÁLISE: VISITANTES QUE NÃO COMPRAM ---\n")
print(f"Número de visitantes que não compram: {visitantes_sem_compra.shape[0]}")

# Estatisticas deste segmento
media_visita_sem_compra = np.mean(visitantes_sem_compra[:, 0])
media_tempo_sem_compra = np.mean(visitantes_sem_compra[:, 1])

print(f"Média de visitas desses visitantes: {media_visita_sem_compra:.2f}")
print(f"Apesar de não comprarem, eles passam em média: {media_tempo_sem_compra:.2f} min no site")

## 5. Análise de Correlação

In [ ]:
# A função mp.corrcoef calcula a matriz de correlação
# rowvar = False indica que as colunas são as variáveis
matriz_correlacao = np.corrcoef(dados_ecommerce, rowvar = False)

print("\n--- MATRIZ DE CORRELAÇÃO ---\n")
print("[Visitas, Tempo, Itens, Valor]\n")
print(np.round(matriz_correlacao, 2))

## 5.1 Análise Gráfica da Matriz de Correlação

In [ ]:
# Calcula a matriz de correlação
matriz_correlacao = np.corrcoef(dados_ecommerce, rowvar = False)

# Define os nomes das variáveis
nomes_variaveis = ["Visitas", "Tempo no Site", "Itens no Carrinho", "Valor da Compra"]

# Converte em DataFrame para exibir com rótulos
df_correlacao = pd.DataFrame(matriz_correlacao,
                            index = nomes_variaveis,
                            columns = nomes_variaveis)

# Matriz de correlação
plt.figure(figsize = (7, 5))
sns.heatmap(df_correlacao, annot = True, cmap = "Blues", fmt = ".2f")
plt.title("Matriz de Correlação")
plt.show